# **InsightViewer — Demo Pipeline**

End-to-end walkthrough of the four pipeline stages:

| # | Stage | Input | Output |
|---|-------|-------|--------|
| 1 | **Ingest** | PDF / Excel / CSV | Chroma (chunks) + DuckDB (metrics) |
| 2 | **Retrieve** | Natural-language query | Ranked text chunks |
| 3 | **Query** | Ticker / metric name | Structured metrics DataFrame |
| 4 | **Visualize** | DataFrames | Matplotlib charts |

A final cell wires all four stages into a single `run_pipeline()` call.

## **Setup**

In [ ]:
from __future__ import annotations

import hashlib, json, os, re, uuid
from pathlib import Path
from typing import Any, Dict, List, Optional

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Vector store
from langchain_core.documents import Document
try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma  # type: ignore
try:
    from langchain_community.embeddings import SentenceTransformerEmbeddings
except ImportError:
    from langchain.embeddings import SentenceTransformerEmbeddings  # type: ignore
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

# Parsers
import pymupdf4llm
import pdfplumber

# Paths
ROOT          = Path("/home/ssever/InsightViewer")
DATA_DIR      = ROOT / "data" / "test"
CHROMA_DIR    = ROOT / "storage" / "chroma"
DUCKDB_PATH   = ROOT / "storage" / "metrics.duckdb"
COLLECTION    = "filings"

# Model / chunking
EMBED_MODEL   = os.getenv("EMBED_MODEL", "all-MiniLM-L12-v2")
CHUNK_SIZE    = int(os.getenv("CHUNK_SIZE",    "900"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "120"))

print("Setup complete.")

Setup complete.


## **Ingest**

Three data sources are handled here:

- **PDF** → markdown → chunks → Chroma; tables → metrics → DuckDB
- **Excel / CSV** → pandas DataFrame (used directly in visualizations)

Helper functions are defined in sub-cells, then applied to the files in `data/test/`.

In [ ]:
# PDF -> Markdown -> Chunks -> Chroma

def pdf_to_markdown(path: Path) -> str:
    return pymupdf4llm.to_markdown(str(path))


def md_to_chunks(md: str, meta: dict) -> List[dict]:
    """Split markdown on headers, then by character limit."""
    header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")])
    chunks: List[dict] = []
    for doc in header_splitter.split_text(md):
        text = doc.page_content
        m    = {**meta, **doc.metadata}
        m["has_table"] = bool(re.search(r"(^|\n)\s*\|.+\|\s*(\n|$)", text))
        if m["has_table"] or len(text) <= CHUNK_SIZE:
            chunks.append({"text": text, "metadata": m})
        else:
            for t in RecursiveCharacterTextSplitter(
                chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
            ).split_text(text):
                chunks.append({"text": t, "metadata": m})
    return chunks


def get_vectorstore() -> Chroma:
    emb = SentenceTransformerEmbeddings(model_name=EMBED_MODEL)
    return Chroma(
        collection_name=COLLECTION,
        persist_directory=str(CHROMA_DIR),
        embedding_function=emb,
    )


def upsert_chunks(chunks: List[dict]) -> int:
    """Add / update chunks in Chroma with stable deterministic IDs."""
    if not chunks:
        return 0
    vs    = get_vectorstore()
    texts, metas, ids = [], [], []
    for i, c in enumerate(chunks):
        m = {
            k: str(v) if not isinstance(v, (str, int, float, bool)) else v
            for k, v in c["metadata"].items() if v is not None
        }
        m["chunk_index"] = i
        texts.append(c["text"])
        metas.append(m)
        ids.append(hashlib.sha1(f"{m.get('source_path','')}::{i}".encode()).hexdigest())
    vs.add_texts(texts=texts, metadatas=metas, ids=ids)
    try:
        vs.persist()  # type: ignore[attr-defined]
    except Exception:
        pass
    return len(texts)


print("PDF -> Chroma helpers ready.")

In [ ]:
# PDF tables -> tidy metrics -> DuckDB

TARGET_METRICS: Dict[str, List[str]] = {
    "revenue":    ["revenue", "net revenue", "total revenue", "net sales"],
    "net_income": ["net income", "net earnings", "profit for the year", "net profit"],
}


def _parse_number(cell: Any) -> Optional[float]:
    s = str(cell).strip().replace(",", "")
    neg = s.startswith("(") and s.endswith(")")
    if neg:
        s = s[1:-1]
    s = re.sub(r"[^\d.\-]", "", s)
    try:
        v = float(s)
        return -v if neg else v
    except (ValueError, TypeError):
        return None

def _best_metric(label: str) -> Optional[str]:
    from rapidfuzz import fuzz, process
    ln = re.sub(r"\s+", " ", label.lower()).strip()
    best_norm, best_score = None, 0
    for norm, syns in TARGET_METRICS.items():
        r = process.extractOne(ln, syns, scorer=fuzz.token_sort_ratio)
        if r and r[1] > best_score:
            best_norm, best_score = norm, r[1]
    return best_norm if best_score >= 80 else None


def extract_metrics(path: Path, meta: dict) -> List[dict]:
    """Extract revenue / net-income rows from PDF tables."""
    md_text = pdf_to_markdown(path)
    units   = "USD" if ("$" in md_text or "usd" in md_text.lower()) else None
    scale   = (
        1_000_000 if "in millions"  in md_text.lower() else
        1_000     if "in thousands" in md_text.lower() else
        None
    )
    rows: List[dict] = []
    with pdfplumber.open(str(path)) as pdf:
        for p_idx, page in enumerate(pdf.pages):
            for t_idx, table in enumerate(page.extract_tables() or []):
                if not table or len(table) < 2:
                    continue
                df  = pd.DataFrame(table)
                hdr = df.iloc[0].astype(str).tolist()
                if sum(bool(re.search(r"[A-Za-z]", c)) for c in hdr) >= 2:
                    df.columns = hdr
                    df = df.iloc[1:].reset_index(drop=True)
                label_col = df.columns[0]
                year_cols = {
                    c: re.search(r"(20\d{2})", str(c)).group(1)
                    for c in df.columns if re.search(r"(20\d{2})", str(c))
                }
                if not year_cols:
                    continue
                for _, row in df.iterrows():
                    label  = str(row[label_col])
                    metric = _best_metric(label)
                    if not metric:
                        continue
                    for col, year in year_cols.items():
                        val = _parse_number(row[col])
                        if val is None:
                            continue
                        rows.append({
                            "id":              str(uuid.uuid4()),
                            "ticker":          meta.get("ticker"),
                            "filing_type":     meta.get("filing_type"),
                            "fiscal_year":     int(meta["fiscal_year"]) if meta.get("fiscal_year") else None,
                            "fiscal_period":   meta.get("fiscal_period"),
                            "metric":          metric,
                            "period_year":     int(year),
                            "value":           float(val * (scale or 1)),
                            "units":           units,
                            "scale":           scale,
                            "source_filename": path.name,
                            "page":            p_idx + 1,
                            "table_id":        t_idx + 1,
                            "provenance":      json.dumps({"label_raw": label}),
                        })
    return rows


# DuckDB helpers
_DDL = """
CREATE TABLE IF NOT EXISTS metrics (
    id TEXT PRIMARY KEY, ticker TEXT, filing_type TEXT,
    fiscal_year INTEGER, fiscal_period TEXT, metric TEXT,
    period_year INTEGER, value DOUBLE, units TEXT, scale INTEGER,
    source_filename TEXT, page INTEGER, table_id INTEGER, provenance JSON
)
"""

def init_db() -> duckdb.DuckDBPyConnection:
    conn = duckdb.connect(str(DUCKDB_PATH))
    conn.execute(_DDL)
    return conn


def insert_metrics(conn: duckdb.DuckDBPyConnection, rows: List[dict]) -> int:
    if not rows:
        return 0
    df = pd.DataFrame(rows)
    conn.register("_rows", df)
    conn.execute("INSERT OR REPLACE INTO metrics SELECT * FROM _rows")
    conn.unregister("_rows")
    return len(rows)


print("Metrics + DuckDB helpers ready.")

In [ ]:
# Filename-based metadata  (ticker / FY / filing type)

def filename_meta(path: Path) -> dict:
    """Derive ticker, fiscal year, period, and form type from the filename."""
    stem   = path.stem  # e.g. 'MSFT_FY25Q1_10Q'
    m_tick = re.match(r"^([A-Z]{1,6})", stem)
    m_fy   = re.search(r"FY(\d{2,4})", stem, re.I)
    m_q    = re.search(r"Q(\d)",        stem, re.I)
    m_form = re.search(r"(10Q|10K|8K)", stem, re.I)
    fy_raw = m_fy.group(1) if m_fy else None
    return {
        "ticker":          m_tick.group(1) if m_tick else None,
        "fiscal_year":     ("20" + fy_raw if fy_raw and len(fy_raw) == 2 else fy_raw),
        "fiscal_period":   f"Q{m_q.group(1)}" if m_q else None,
        "filing_type":     {"10Q": "10-Q", "10K": "10-K", "8K": "8-K"}.get(
                               m_form.group(1).upper()) if m_form else None,
        "source_path":     str(path),
        "source_filename": path.name,
    }


# Run ingestion on every PDF in DATA_DIR

conn = init_db()

pdf_files = sorted(DATA_DIR.glob("*.pdf"))
if not pdf_files:
    print(f"No PDFs found in {DATA_DIR}")
else:
    for pdf_path in pdf_files:
        meta = filename_meta(pdf_path)
        print(f"Ingesting  {pdf_path.name}")
        print(f"  meta   : {meta}")

        md        = pdf_to_markdown(pdf_path)
        chunks    = md_to_chunks(md, meta)
        n_chunks  = upsert_chunks(chunks)

        rows      = extract_metrics(pdf_path, meta)
        n_metrics = insert_metrics(conn, rows)

        print(f"  result : {n_chunks} chunks upserted  |  {n_metrics} metric rows inserted\n")

total = conn.execute("SELECT COUNT(*) FROM metrics").fetchone()[0]
print(f"DuckDB total metric rows: {total}")

In [ ]:
# Load spreadsheet / CSV data with pandas

# Dividend history (Excel)
div_path = DATA_DIR / "Microsofts-Dividend-History.xlsx"
df_div   = pd.read_excel(div_path) if div_path.exists() else pd.DataFrame()
if not df_div.empty:
    df_div.columns = [c.strip().lower().replace(" ", "_") for c in df_div.columns]
    print(f"Dividend data : {df_div.shape[0]} rows, columns: {list(df_div.columns)}")
    display(df_div.head())

# Stock price history (CSV)
csv_path = ROOT / "data" / "sql" / "MSFT_1986-03-13_2025-02-04.csv"
df_stock = pd.read_csv(csv_path, parse_dates=["Date"]) if csv_path.exists() else pd.DataFrame()
if not df_stock.empty:
    print(f"\nStock data : {df_stock.shape[0]} rows  |  "
          f"{df_stock['Date'].min().date()} -> {df_stock['Date'].max().date()}")
    display(df_stock.tail())

## **Retrieve — Semantic Search**

Ask a natural-language question and get back the most relevant text chunks from the vector store.

In [ ]:
# Example query + similarity search on vector store

USER_QUERY = "What drove Microsoft's revenue growth in Q1 FY2025?"
K          = 5

vs      = get_vectorstore()
results = vs.similarity_search_with_score(USER_QUERY, k=K)

print(f"Query : {USER_QUERY}")
print("-" * 70)
for rank, (doc, score) in enumerate(results, start=1):
    m      = doc.metadata
    header = " / ".join(filter(None, [m.get("h1"), m.get("h2"), m.get("h3")]))
    print(f"[{rank}]  score={score:.4f}  |  {m.get('source_filename', '')}  |  {header}")
    print(doc.page_content[:300].replace("\n", " "))
    print()

## **Query — Structured Metrics (DuckDB)**

Pull revenue and net income from the metrics table and aggregate by year.

In [ ]:
# Querying extracted metrics in DuckDB

TICKER = "MSFT"

df_metrics = conn.execute("""
    SELECT
        ticker,
        metric,
        period_year,
        ROUND(SUM(value) / 1e9, 2) AS value_billions
    FROM   metrics
    WHERE  ticker = ?
    GROUP  BY ticker, metric, period_year
    ORDER  BY metric, period_year
""", [TICKER]).df()

if df_metrics.empty:
    print("No metrics in DuckDB yet -- re-run Section 1 to ingest a PDF with financial tables.")
else:
    print(f"Metrics for {TICKER} ({df_metrics['period_year'].nunique()} distinct years):")
    display(df_metrics.pivot(index="period_year", columns="metric", values="value_billions"))

## **Visualize**

Three charts in one figure:
1. Revenue & Net Income trend (from DuckDB)
2. Dividend-per-share history (from Excel)
3. Adjusted close price, last 5 years (from CSV)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Microsoft (MSFT) -- Financial Overview", fontsize=14, fontweight="bold")

# Chart 1: Revenue & Net Income
ax1 = axes[0]
if not df_metrics.empty:
    for metric, label, color, offset in [
        ("revenue",    "Revenue",    "#0078D4", -0.2),
        ("net_income", "Net Income", "#50C878",  0.2),
    ]:
        sub = df_metrics[df_metrics["metric"] == metric].sort_values("period_year")
        if not sub.empty:
            ax1.bar(
                sub["period_year"] + offset,
                sub["value_billions"],
                width=0.35, label=label, color=color, alpha=0.85,
            )
    ax1.set_title("Revenue & Net Income")
    ax1.set_xlabel("Year")
    ax1.set_ylabel("USD (billions)")
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}B"))
    ax1.legend()
else:
    ax1.text(0.5, 0.5, "No metrics data\n(ingest a PDF first)",
             ha="center", va="center", transform=ax1.transAxes, color="gray")
    ax1.set_title("Revenue & Net Income")

# Chart 2: Dividend history
ax2 = axes[1]
if not df_div.empty:
    date_col = next((c for c in df_div.columns if "date"   in c), None)
    amt_col  = next((c for c in df_div.columns if "amount" in c or "dividend" in c), None)
    if date_col and amt_col:
        df_d = df_div[[date_col, amt_col]].copy()
        df_d[date_col] = pd.to_datetime(df_d[date_col], errors="coerce")
        df_d[amt_col]  = pd.to_numeric(df_d[amt_col],   errors="coerce")
        df_d = df_d.dropna().sort_values(date_col)
        ax2.bar(df_d[date_col], df_d[amt_col], width=60, color="#FFB900", alpha=0.85)
        ax2.set_title("Dividend per Share")
        ax2.set_xlabel("Date")
        ax2.set_ylabel("Amount ($)")
        ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.2f}"))
else:
    ax2.text(0.5, 0.5, "No dividend data", ha="center", va="center",
             transform=ax2.transAxes, color="gray")
    ax2.set_title("Dividend per Share")

# Chart 3: Stock price (2020 -> latest)
ax3 = axes[2]
if not df_stock.empty:
    df_recent = df_stock[df_stock["Date"] >= "2020-01-01"].copy()
    ax3.plot(df_recent["Date"], df_recent["Adj Close"], color="#0078D4", linewidth=1.2)
    ax3.fill_between(df_recent["Date"], df_recent["Adj Close"], alpha=0.12, color="#0078D4")
    ax3.set_title("Stock Price (2020-2025)")
    ax3.set_xlabel("Date")
    ax3.set_ylabel("Adj. Close ($)")
    ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}"))
else:
    ax3.text(0.5, 0.5, "No stock data", ha="center", va="center",
             transform=ax3.transAxes, color="gray")
    ax3.set_title("Stock Price")

plt.tight_layout()
out_path = ROOT / "storage" / "demo_charts.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved -> {out_path}")

## **End-to-End Pipeline**

`run_pipeline()` accepts a free-text user request and a ticker, then:

1. Retrieves the top-*k* relevant text chunks from Chroma (semantic search)
2. Fetches structured metrics from DuckDB
3. Attaches the spreadsheet DataFrames loaded above
4. Returns a `context` dict ready for an LLM synthesis call and/or charting

In [ ]:
# Orchestrate retrieval for a single user request, combining all sources

def run_pipeline(
    user_request: str,
    ticker: str = "MSFT",
    k: int      = 5,
) -> dict:
    """
    Orchestrate retrieval for a single user request.

    Returns
    -------
    context : dict
        user_request  -- original query string
        text_chunks   -- list[(Document, score)] from Chroma
        text_context  -- pre-formatted string for LLM prompt injection
        metrics_df    -- DataFrame of aggregated metrics from DuckDB
        df_div        -- dividend history DataFrame (may be empty)
        df_stock      -- stock price DataFrame      (may be empty)
    """
    print(f"Request : {user_request}")
    print("-" * 70)

    # Semantic retrieval
    vs     = get_vectorstore()
    chunks = vs.similarity_search_with_score(user_request, k=k)
    text_context = "\n\n".join(
        f"[score={s:.3f} | {d.metadata.get('source_filename', '')}]\n{d.page_content[:500]}"
        for d, s in chunks
    )
    print(f"[1] {len(chunks)} text chunks retrieved from Chroma")

    # Structured metrics
    df_m = conn.execute("""
        SELECT metric, period_year, ROUND(SUM(value)/1e9, 2) AS value_billions
        FROM   metrics
        WHERE  ticker = ?
        GROUP  BY metric, period_year
        ORDER  BY metric, period_year
    """, [ticker]).df()
    print(f"[2] {df_m.shape[0]} metric rows fetched from DuckDB")

    # Spreadsheet data (loaded in Section 1)
    print(f"[3] Dividend rows: {len(df_div)}  |  Stock price rows: {len(df_stock)}")

    context = {
        "user_request": user_request,
        "text_chunks":  chunks,
        "text_context": text_context,
        "metrics_df":   df_m,
        "df_div":       df_div,
        "df_stock":     df_stock,
    }

    # -- LLM synthesis stub --------------------------
    from google import genai
    client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=(
            f"User question: {user_request}\n\n"
            f"Retrieved context:\n{text_context}\n\n"
            f"Financial metrics (USD billions):\n{df_m.to_string()}\n\n"
            "Provide a concise analytical answer."
        )
    )
    context["llm_answer"] = response.text
    print(f"\nLLM answer:\n{context['llm_answer']}")
    # -------------------------------------------------------------------------

    return context


# Example run of the full retrieval pipeline for a user query, combining all sources
ctx = run_pipeline("What drove Microsoft's revenue growth in Q1 FY2025?")

print("\nTop chunk preview:")
print(ctx["text_chunks"][0][0].page_content[:400])

if not ctx["metrics_df"].empty:
    print("\nMetrics pivot:")
    display(ctx["metrics_df"].pivot(index="period_year", columns="metric", values="value_billions"))